In [275]:
import scanpy as sc
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder
import xgboost

In [276]:
np.random.seed(42)

In [277]:
# path = "/project/Wellcome_Discovery/datashare/sriva/GENESIS/"
path = "data/retina/"

In [278]:
rna = sc.read_h5ad(path+"retina_sc_filt.h5ad")
gex = sc.read_h5ad(path+"retina_sn_filt.h5ad")
fake = sc.read_h5ad(path+"retina_Fake_RNA_VAE_UNET_log_norm.h5ad")

In [279]:
marker_genes = {
    "retinal cone cell": [
        "ARR3", "GNAT2", "PDE6C", "CNGA3", "CNGB3", "OPN1SW"
        # optional if present: "OPN1LW", "OPN1MW"
    ],
    "amacrine cell": [
        "GAD1", "GAD2", "SLC6A1", "TFAP2A", "TFAP2B", "PAX6"
        # optional: "SLC18A2" (some amacrines), "CHAT" (starburst subtype)
    ],
    "Mueller cell": [
        "RLBP1", "GLUL", "SLC1A3", "VIM", "SOX9", "AQP4"
    ],
    "retinal ganglion cell": [
        "RBPMS", "SNCG", "POU4F1", "POU4F2", "NEFL", "NEFM"
    ],
    "retinal rod cell": [
        "RHO", "GNAT1", "PDE6A", "PDE6B", "NRL", "SAG", "CNGA1"
    ],
    "astrocyte": [
        "GFAP", "ALDH1L1", "AQP4", "SLC1A3", "SOX9"
    ],
    "retina horizontal cell": [
        "PROX1", "ONECUT1", "ONECUT2", "LHX1"
    ],
}

In [280]:
genes = []

for row in list(marker_genes.values()):
    for item in row:
        if item not in genes:
            genes.append(item)
            
np.random.shuffle(genes)

In [281]:
df_rna = pd.DataFrame(index   = rna.obs.index,
                      columns = genes,
                      data    = rna.X[:, rna.var.feature_name.isin(genes)])

df_gex = pd.DataFrame(index   = gex.obs.index,
                      columns = genes,
                      data    = gex.X[:, gex.var.feature_name.isin(genes)])

df_fake = pd.DataFrame(index   = fake.obs.index,
                       columns = genes,
                       data    = fake.X[:, fake.var.feature_name.isin(genes)])

In [282]:
# Encode labels
label_encoder = LabelEncoder()
label_encoder.fit(rna.obs["cell_type"].values.tolist())
label_encoder.classes_

array(['Mueller cell', 'amacrine cell', 'astrocyte',
       'retina horizontal cell', 'retinal cone cell',
       'retinal ganglion cell', 'retinal rod cell'], dtype='<U22')

In [283]:
df_rna["cell_type"]  = label_encoder.transform(rna.obs["cell_type"].values.tolist())
df_gex["cell_type"]  = label_encoder.transform(gex.obs["cell_type"].values.tolist())
df_fake["cell_type"] = label_encoder.transform(rna.obs["cell_type"].values.tolist())

In [284]:
# Create train-test split (final evaluation set)
X_train_rna, X_test_rna, y_train_rna, y_test_rna = train_test_split(df_rna[df_rna.columns[:-1]],
                                                                    df_rna[df_rna.columns[-1]],
                                                                    test_size=0.15,
                                                                    random_state=42,
                                                                    stratify=df_rna[df_rna.columns[-1]]
                                                                    )

In [285]:
# Create train-test split (final evaluation set)
X_train_gex, X_test_gex, y_train_gex, y_test_gex = train_test_split(df_gex[df_gex.columns[:-1]],
                                                                    df_gex[df_gex.columns[-1]],
                                                                    test_size=0.15,
                                                                    random_state=42,
                                                                    stratify=df_gex[df_gex.columns[-1]]
                                                                    )

In [286]:
# Create train-test split (final evaluation set)
X_train_fake, X_test_fake, y_train_fake, y_test_fake = train_test_split(df_fake[df_fake.columns[:-1]],
                                                                        df_fake[df_fake.columns[-1]],
                                                                        test_size=0.15,
                                                                        random_state=42,
                                                                        stratify=df_fake[df_fake.columns[-1]]
                                                                        )

# Training and hyperparameter setting xgboost on scRNA

In [287]:
# Define XGBoost classifier
xgb_clf = xgboost.XGBClassifier(objective='multi:softprob',
                             num_class=len(label_encoder.classes_),
                             device="cpu",
                             random_state=42)

In [288]:
# Hyperparameter grid
param_grid = {'booster': ['gbtree', 'dart'],
              'max_depth': [2, 4, 6],
              'learning_rate': [0.01, 0.05, 0.1],
              'n_estimators': [50, 100, 150],
              'subsample': [0.6, 0.8, 1.0],
              }

In [289]:
# Cross-validation and hyperparameter tuning
clf_cv = GridSearchCV(estimator=xgb_clf,
                      param_grid=param_grid,
                      scoring='accuracy',
                      cv=5,
                      verbose=1,
                      n_jobs=8)

clf_cv.fit(X_train_rna, y_train_rna)

Fitting 5 folds for each of 162 candidates, totalling 810 fits
[16:31:49] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:50] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:51] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:53] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:54] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:55] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:57] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:58] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:32:00] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:32:02] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:32:05] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:32:08] WA

[16:31:49] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:50] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:50] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:51] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:53] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:54] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:56] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:58] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:59] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:32:02] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:32:05] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:32:09] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not

[16:31:49] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:50] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:51] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:52] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:54] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:56] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:57] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:58] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:32:00] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:32:02] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:32:04] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:32:08] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not

[16:31:49] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:50] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:51] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:53] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:54] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:56] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:58] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:59] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:32:03] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:32:06] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:32:10] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:32:12] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not

[16:31:49] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:51] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:52] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:54] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:56] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:58] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:59] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:32:00] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:32:03] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:32:06] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:32:10] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:32:14] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not

[16:31:49] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:50] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:51] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:53] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:54] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:56] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:58] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:59] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:32:00] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:32:03] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:32:06] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:32:10] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not

[16:45:24] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.



,estimator,"XGBClassifier...ree=None, ...)"
,param_grid,"{'booster': ['gbtree', 'dart'], 'learning_rate': [0.01, 0.05, ...], 'max_depth': [2, 4, ...], 'n_estimators': [50, 100, ...], ...}"
,scoring,'accuracy'
,n_jobs,8
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,objective,'multi:softprob'


In [290]:
# Best hyperparameters
print("Best hyperparameters:")
print(clf_cv.best_params_)

# Evaluate on final test set
best_model = clf_cv.best_estimator_

Best hyperparameters:
{'booster': 'gbtree', 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 150, 'subsample': 0.6}


In [ ]:
# Best hyperparameters:
# {'booster': 'gbtree', 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 150, 'subsample': 0.6}

In [297]:
best_model.save_model('retina_best_xgb_model.json')

# Performance evaluation on scRNA, scGEX, generated scGEX

In [298]:
model_rna = xgboost.XGBClassifier()
model_rna.load_model('retina_best_xgb_model.json')

In [299]:
# y_pred_rna = model_rna.predict(X_train_rna)
# print(f'Accuracy: {accuracy_score(y_train_rna.values, y_pred_rna):.4f}')

In [300]:
y_pred_rna = model_rna.predict(X_test_rna)
print(f'Accuracy: {accuracy_score(y_test_rna.values, y_pred_rna):.4f}')

Accuracy: 0.9817


In [301]:
y_pred_gex = model_rna.predict(X_test_gex)
print(f'Accuracy: {accuracy_score(y_test_gex.values, y_pred_gex):.4f}')

Accuracy: 0.1308


In [302]:
y_pred_fake = best_model.predict(X_test_fake)
print(f'Accuracy: {accuracy_score(y_test_fake, y_pred_fake):.4f}')

Accuracy: 0.3080


# Fine-tuning on scGEX

In [304]:
model_gex = xgboost.XGBClassifier()
model_gex.load_model('retina_best_xgb_model.json')

In [305]:
model_gex.fit(X_train_gex, 
              y_train_gex,
              xgb_model=model_gex.get_booster(),  # Continue training from existing model
              verbose=True
              )

[16:47:46] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.



,objective,'multi:softprob'
,use_label_encoder,None
,base_score,None
,booster,'gbtree'
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [306]:
y_pred_rna = model_gex.predict(X_test_rna)
print(f'Accuracy: {accuracy_score(y_test_rna.values, y_pred_rna):.4f}')

Accuracy: 0.6920


In [307]:
y_pred_gex = model_gex.predict(X_test_gex)
print(f'Accuracy: {accuracy_score(y_test_gex.values, y_pred_gex):.4f}')

Accuracy: 0.9612


In [308]:
y_pred_fake = model_gex.predict(X_test_fake)
print(f'Accuracy: {accuracy_score(y_test_fake, y_pred_fake):.4f}')

Accuracy: 0.1445


# Fine-tuning on generated scRNA

In [309]:
model_fake = xgboost.XGBClassifier()
model_fake.load_model('retina_best_xgb_model.json')

In [310]:
model_fake.fit(X_train_fake, 
               y_train_fake,
               xgb_model=model_fake.get_booster(),  # Continue training from existing model
               verbose=True
               )

[16:48:29] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.



,objective,'multi:softprob'
,use_label_encoder,None
,base_score,None
,booster,'gbtree'
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [311]:
y_pred_rna = model_fake.predict(X_test_rna)
print(f'Accuracy: {accuracy_score(y_test_rna.values, y_pred_rna):.4f}')

Accuracy: 0.8433


In [312]:
y_pred_gex = model_fake.predict(X_test_gex)
print(f'Accuracy: {accuracy_score(y_test_gex.values, y_pred_gex):.4f}')

Accuracy: 0.1027


In [313]:
y_pred_fake = model_fake.predict(X_test_fake)
print(f'Accuracy: {accuracy_score(y_test_fake, y_pred_fake):.4f}')

Accuracy: 0.9810
[16:31:50] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:51] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:53] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:54] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:56] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:58] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:31:59] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:32:00] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:32:03] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:32:08] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:32:12] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:32:13] WARNING: ../src/learner.cc:767: 
Parameters: { "


[16:41:38] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:41:48] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:41:57] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:42:17] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:42:38] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:42:41] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:42:52] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:43:03] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:43:26] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:43:49] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:43:53] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:44:06] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are no


[16:42:52] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:43:03] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:43:24] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:43:48] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:43:52] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:43:56] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:44:08] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:44:20] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:44:46] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.


[16:42:45] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:42:55] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are not used.

[16:43:06] WARNING: ../src/learner.cc:767: 
Parameters: { "device" } are n